In [7]:
import os
import re
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

In [30]:
def load_and_split_chapters(folder_path, book_name):
    """
    Load all text files from a folder and split them into chapters.
    
    Args:
        folder_path: Path to the folder containing text files
        book_name: Name of the book (e.g., "Ganesh_Puran_Upasana_Khand")
    
    Returns:
        List of Document objects, each representing a chapter
    """
    documents = []
    
    # Get all text files in the folder
    text_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    for file_name in text_files:
        file_path = os.path.join(folder_path, file_name)
        
        # Load the document
        loader = TextLoader(file_path, encoding='utf-8')
        loaded_docs = loader.load()
        
        # Since TextLoader returns a list, get the first (and only) document
        full_text = loaded_docs[0].page_content
        
        # Split by chapter headings
        # Pattern to match chapter headings like "Chapter 1 : The Description of Somakanta"
        # Also handles variations like "Chapter 1" or "Chapter 1."
        chapter_pattern = r'(Chapter\s+\d+\s*[:.]?\s*[^\n]+)'
        
        # Find all chapter headings with their positions
        chapter_matches = list(re.finditer(chapter_pattern, full_text, re.IGNORECASE))
        
        if not chapter_matches:
            # If no chapters found, treat the whole file as one document
            doc = Document(
                page_content=full_text,
                metadata={
                    "book": book_name,
                    "file": file_name,
                    "chapter_info": "Full text"
                }
            )
            documents.append(doc)
        else:
            # Split the text by chapters
            for i, match in enumerate(chapter_matches):
                # Get the chapter heading
                chapter_heading = match.group(1).strip()
                
                # Determine the start and end of the chapter content
                start_pos = match.start()
                
                # If this is the last chapter, go to the end of text
                if i == len(chapter_matches) - 1:
                    end_pos = len(full_text)
                else:
                    end_pos = chapter_matches[i + 1].start()
                
                # Extract chapter content
                chapter_content = full_text[start_pos:end_pos].strip()
                
                # Create a Document object for this chapter
                doc = Document(
                    page_content=chapter_content,
                    metadata={
                        "book": book_name,
                        "file": file_name,
                        "chapter_info": chapter_heading
                    }
                )
                documents.append(doc)
    
    return documents

In [31]:
def load_and_split_chapters_for_file(file_path, book_name):
    """
    Helper function to load and split a single file into chapters.
    """
    documents = []
    
    # Load the document
    loader = TextLoader(file_path, encoding='utf-8')
    loaded_docs = loader.load()
    full_text = loaded_docs[0].page_content
    
    # Split by chapter headings
    chapter_pattern = r'(Chapter\s+\d+\s*[:.]?\s*[^\n]+)'
    chapter_matches = list(re.finditer(chapter_pattern, full_text, re.IGNORECASE))
    
    if not chapter_matches:
        # If no chapters found, treat the whole file as one document
        doc = Document(
            page_content=full_text,
            metadata={
                "book": book_name,
                "file": os.path.basename(file_path),
                "chapter_info": "Full text"
            }
        )
        documents.append(doc)
    else:
        # Split the text by chapters
        for i, match in enumerate(chapter_matches):
            chapter_heading = match.group(1).strip()
            start_pos = match.start()
            
            if i == len(chapter_matches) - 1:
                end_pos = len(full_text)
            else:
                end_pos = chapter_matches[i + 1].start()
            
            chapter_content = full_text[start_pos:end_pos].strip()
            
            doc = Document(
                page_content=chapter_content,
                metadata={
                    "book": book_name,
                    "file": os.path.basename(file_path),
                    "chapter_info": chapter_heading
                }
            )
            documents.append(doc)
    
    return documents

In [32]:
ganesh_folder = "docs/ganesh_puran"
ganesh_files = ["Ganesh_Puran_Krida_Khand.txt", "Ganesh_Puran_Upasana_Khand.txt"]

all_documents = []

for file_name in ganesh_files:
    file_path = os.path.join(ganesh_folder, file_name)
    if os.path.exists(file_path):
        # Extract book name from file name (remove .txt extension)
        book_name = file_name.replace('.txt', '')
        # Load and split chapters from this file
        docs = load_and_split_chapters_for_file(file_path, book_name)
        all_documents.extend(docs)

# Load documents from Mudgal Puran folder
mudgal_folder = "docs/mudgal"
mudgal_files = [f"Mudgal_Puran_Khand_{i}.txt" for i in range(1, 10)]

for file_name in mudgal_files:
    file_path = os.path.join(mudgal_folder, file_name)
    if os.path.exists(file_path):
        book_name = file_name.replace('.txt', '')
        docs = load_and_split_chapters_for_file(file_path, book_name)
        all_documents.extend(docs)

In [33]:
print(f"Total documents loaded: {len(all_documents)}")

Total documents loaded: 675


In [34]:
print("\nSample document metadata:")
if all_documents:
    sample_doc = all_documents[152]
    print(f"Book: {sample_doc.metadata['book']}")
    print(f"File: {sample_doc.metadata['file']}")
    print(f"Chapter: {sample_doc.metadata['chapter_info']}")
    print(f"Content preview: {sample_doc.page_content[:100]}...")


Sample document metadata:
Book: Ganesh_Puran_Krida_Khand
File: Ganesh_Puran_Krida_Khand.txt
Chapter: Chapter 153 : The Description of Somakanta Attaining the Abode of the Divine
Content preview: Chapter 153 : The Description of Somakanta Attaining the Abode of the Divine

Suta said: The four mi...


In [28]:
def count_chapters_in_file(file_path):
    """Count number of chapters in a single file"""
    loader = TextLoader(file_path, encoding='utf-8')
    docs = loader.load()
    full_text = docs[0].page_content
    
    chapter_pattern = r'(Chapter\s+\d+\s*[:.]?\s*[^\n]+)'
    matches = re.findall(chapter_pattern, full_text, re.IGNORECASE)
    
    return len(matches) if matches else 1  # If no chapters, count as 1

# Ganesh Puran files
ganesh_folder = "docs/ganesh_puran"
ganesh_files = ["Ganesh_Puran_Krida_Khand.txt", "Ganesh_Puran_Upasana_Khand.txt"]

print("=" * 50)
print("GANESH PURAN")
print("=" * 50)
for file in ganesh_files:
    file_path = os.path.join(ganesh_folder, file)
    if os.path.exists(file_path):
        count = count_chapters_in_file(file_path)
        print(f"{file}: {count} chapters")

print("\n" + "=" * 50)
print("MUDGAL PURAN")
print("=" * 50)

# Mudgal Puran files
mudgal_folder = "docs/mudgal"
mudgal_files = [f"Mudgal_Puran_Khand_{i}.txt" for i in range(1, 10)]

total_chapters = 0
for file in mudgal_files:
    file_path = os.path.join(mudgal_folder, file)
    if os.path.exists(file_path):
        count = count_chapters_in_file(file_path)
        total_chapters += count
        print(f"{file}: {count} chapters")

print("\n" + "=" * 50)
print(f"TOTAL CHAPTERS ACROSS ALL BOOKS: {total_chapters}")
print("=" * 50)

GANESH PURAN
Ganesh_Puran_Krida_Khand.txt: 155 chapters
Ganesh_Puran_Upasana_Khand.txt: 92 chapters

MUDGAL PURAN
Mudgal_Puran_Khand_1.txt: 54 chapters
Mudgal_Puran_Khand_2.txt: 74 chapters
Mudgal_Puran_Khand_3.txt: 51 chapters
Mudgal_Puran_Khand_4.txt: 52 chapters
Mudgal_Puran_Khand_5.txt: 45 chapters
Mudgal_Puran_Khand_6.txt: 45 chapters
Mudgal_Puran_Khand_7.txt: 16 chapters
Mudgal_Puran_Khand_8.txt: 50 chapters
Mudgal_Puran_Khand_9.txt: 41 chapters

TOTAL CHAPTERS ACROSS ALL BOOKS: 428
